In [1]:
import torch
import time
import re
from PIL import Image
from pathlib import Path
from transformers import MarianMTModel, MarianTokenizer
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Opus-MT
opus_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ar-de")
opus_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-ar-de").to(device)
print("Opus-MT loaded.")

# Surya
foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()
print("Surya loaded.")

Opus-MT loaded.
Surya loaded.


In [5]:
image_path = Path("../data/inputs/test_arabic_01.png")
image = Image.open(image_path)

# OCR
predictions = recognition_predictor([image], det_predictor=detection_predictor)
arabic_lines = [line.text for line in predictions[0].text_lines]

# Reassemble: join lines, split on sentence terminators only
arabic_continuous = " ".join(arabic_lines)
arabic_sentences = re.split(r'(?<=[.؟!])\s+', arabic_continuous)
arabic_sentences = [s.strip() for s in arabic_sentences if s.strip()]

print(f"{len(arabic_lines)} OCR lines -> {len(arabic_sentences)} reassembled sentences")

Recognizing Text: 100%|██████████████████████████████████████████████████████████████████| 8/8 [00:11<00:00,  1.40s/it]

8 OCR lines -> 5 reassembled sentences


In [6]:
print("=== Opus-MT on reassembled legal text ===\n")
start = time.time()

for i, ar_sent in enumerate(arabic_sentences, 1):
    inputs = opus_tokenizer(ar_sent, return_tensors="pt", padding=True).to(device)
    out = opus_model.generate(**inputs, max_length=512)
    de_sent = opus_tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"Sentence {i}:")
    print(f"  AR: {ar_sent}")
    print(f"  DE: {de_sent}")
    print()

elapsed = time.time() - start
print(f"Translation: {elapsed:.2f}s for {len(arabic_sentences)} sentences")

=== Opus-MT on reassembled legal text ===

Sentence 1:
  AR: القانون هو مجموعة القواعد القانونية التي تهدف إلى تنظيم سلوك الأفراد في المجتمع وتكون مصحوبة بجزاء توقعه السلطة العامة على المخالف عند الاقتضاء.
  DE: Das Recht ist die Gruppe von Rechtsnormen, die darauf abzielen, das Verhalten von Einzelpersonen in der Gesellschaft zu organisieren und gegebenenfalls mit der Erwartung einer Verletzung durch die öffentliche Behörde bestraft zu werden.

Sentence 2:
  AR: إذاً فالقانون علم اجتماعي، موضوعه الإنسان وسلوكه مع نظائره، أعماله وردود أفعاله، وهذا موضوع ضخم، متغير المضمون، غير معروف على وجه التحديد ويصعب عرضه بدقة في أغلب الأحوال، وهدفه حكم الجماعات الإنسانية، حتى لا تترك العلاقات بين الناس، عائلية أو اقتصادية أو سياسية، فوضى ينظمها كل فرد وفق رغبته ومشيئته، وإلا صدقت وتحققت مقولة الفيلسوف بسوت Bossuet: "حيث يملك الكل فعل ما يشاؤون لا يملك أحد فعل ما يشاء، وحيث لا سيد، فالكل سيد، وحيث الكل سيد فالكل عبيد".
  DE: Das Gesetz ist also eine soziale Wissenschaft, das menschliche Thema und s